# EDA — Pima Indians Diabetes

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from scipy import stats

from pipeline import load_diabetes, train_test_split, DIABETES_FEATURES as FEATURE_NAMES
from pipeline import train, score as model_score, predict
from pipeline.data_diabetes import CSV_PATH

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## Load data

In [ ]:
X, Xi, y, Phi, feature_names, col_means, col_stds = load_diabetes()
train_idx, test_idx = train_test_split(len(y))

df_raw = pd.read_csv(CSV_PATH)
X_raw  = df_raw[FEATURE_NAMES].values.astype(float)
y_raw  = np.where(df_raw['Outcome'].values == 1, 1, -1)

print(f'samples   : {len(y)}')
print(f'features  : {len(feature_names)}')
print(f'diabetic  : {(y==-1).sum()}  ({100*(y==-1).mean():.1f}%)')
print(f'not diab  : {(y==1).sum()}  ({100*(y==1).mean():.1f}%)')
print(f'train/test: {len(train_idx)} / {len(test_idx)}')

## 1. Missingness per feature

In [3]:
miss_counts = Xi.sum(axis=0)
miss_pcts   = 100 * Xi.mean(axis=0)
miss_cols   = [j for j in range(len(feature_names)) if miss_pcts[j] > 0]
miss_names  = [feature_names[j] for j in miss_cols]

df_miss = pd.DataFrame({
    'n_missing': miss_counts,
    'pct_missing': miss_pcts.round(2),
}, index=feature_names)
df_miss

,n_missing,pct_missing
Pregnancies,0,0.0000
Glucose,5,0.6500
BloodPressure,35,4.5600
SkinThickness,227,29.5600
Insulin,374,48.7000
BMI,11,1.4300
DiabetesPedigreeFunction,0,0.0000
Age,0,0.0000


## 2. Missingness co-occurrence & pattern counts

In [4]:
# co-occurrence table
cooccur = Xi[:, miss_cols].T @ Xi[:, miss_cols]
np.fill_diagonal(cooccur, 0)
print('Co-occurrence of missingness (off-diagonal = both missing):')
display(pd.DataFrame(cooccur, index=miss_names, columns=miss_names))

# how many features are missing per person
k_missing = Xi.sum(axis=1)
print()
print('# missing features per person:')
counts = {k: int((k_missing == k).sum()) for k in range(int(k_missing.max())+1)}
df_k = pd.DataFrame(list(counts.items()), columns=['k_missing', 'n_persons'])
df_k['pct'] = (df_k['n_persons'] / len(y) * 100).round(1)
display(df_k)
print(f'persons with >= 1 missing: {(k_missing>0).sum()}  ({100*(k_missing>0).mean():.1f}%)')

Co-occurrence of missingness (off-diagonal = both missing):


,Glucose,BloodPressure,SkinThickness,Insulin,BMI
Glucose,0,0,0,4,0
BloodPressure,0,0,33,35,7
SkinThickness,0,33,0,227,9
Insulin,4,35,227,0,10
BMI,0,7,9,10,0



# missing features per person:


,k_missing,n_persons,pct
0,0,392,51.0000
1,1,142,18.5000
2,2,199,25.9000
3,3,28,3.6000
4,4,7,0.9000


persons with >= 1 missing: 376  (49.0%)


## 3. Summary statistics & t-tests by class

In [5]:
rows = []
for j, name in enumerate(feature_names):
    obs  = X_raw[:, j] != 0
    all_ = X_raw[obs, j]
    a    = X_raw[obs & (y_raw ==  1), j]
    b    = X_raw[obs & (y_raw == -1), j]
    _, p = stats.ttest_ind(a, b)
    rows.append({
        'feature'        : name,
        'mean_all'       : round(all_.mean(), 2),
        'std_all'        : round(all_.std(),  2),
        'min'            : round(all_.min(),  2),
        'max'            : round(all_.max(),  2),
        'skew'           : round(stats.skew(all_), 2),
        'mean_diabetic'  : round(a.mean(), 2),
        'mean_not'       : round(b.mean(), 2),
        'diff'           : round(a.mean() - b.mean(), 2),
        'p_value'        : round(p, 4),
    })

pd.DataFrame(rows).set_index('feature')

,mean_all,std_all,min,max,skew,mean_diabetic,mean_not,diff,p_value
feature,,,,,,,,,
Pregnancies,4.4900,3.2100,1.0000,17.0000,0.8800,5.6700,3.8600,1.8100,0.0000
Glucose,121.6900,30.5200,44.0000,199.0000,0.5300,142.3200,110.6400,31.6800,0.0000
BloodPressure,72.4100,12.3700,24.0000,122.0000,0.1300,75.3200,70.8800,4.4400,0.0000
SkinThickness,29.1500,10.4700,7.0000,99.0000,0.6900,33.0000,27.2400,5.7600,0.0000
Insulin,155.5500,118.6300,14.0000,846.0000,2.1600,206.8500,130.2900,76.5600,0.0000
BMI,32.4600,6.9200,18.2000,67.1000,0.5900,35.4100,30.8600,4.5500,0.0000
DiabetesPedigreeFunction,0.4700,0.3300,0.0800,2.4200,1.9200,0.5500,0.4300,0.1200,0.0000
Age,33.2400,11.7500,21.0000,81.0000,1.1300,37.0700,31.1900,5.8800,0.0000


## 4. Correlation matrix (fully-observed rows, standardized)

In [6]:
obs_mask = Xi.sum(axis=1) == 0
corr = np.corrcoef(X[obs_mask].T)
pd.DataFrame(corr.round(3), index=feature_names, columns=feature_names)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
Pregnancies,1.0000,0.1980,0.2130,0.0930,0.0790,-0.0250,0.0080,0.6800
Glucose,0.1980,1.0000,0.2100,0.1990,0.5810,0.2100,0.1400,0.3440
BloodPressure,0.2130,0.2100,1.0000,0.2330,0.0990,0.3040,-0.0160,0.3000
SkinThickness,0.0930,0.1990,0.2330,1.0000,0.1820,0.6640,0.1600,0.1680
Insulin,0.0790,0.5810,0.0990,0.1820,1.0000,0.2260,0.1360,0.2170
BMI,-0.0250,0.2100,0.3040,0.6640,0.2260,1.0000,0.1590,0.0700
DiabetesPedigreeFunction,0.0080,0.1400,-0.0160,0.1600,0.1360,0.1590,1.0000,0.0850
Age,0.6800,0.3440,0.3000,0.1680,0.2170,0.0700,0.0850,1.0000


## 5. Missingness vs outcome (chi-square)

In [ ]:
rows = []
for j in miss_cols:
    name = feature_names[j]
    m_d  = int(((Xi[:,j]==1) & (y==-1)).sum())
    m_nd = int(((Xi[:,j]==1) & (y== 1)).sum())
    o_d  = int(((Xi[:,j]==0) & (y==-1)).sum())
    o_nd = int(((Xi[:,j]==0) & (y== 1)).sum())
    chi2, p, _, _ = stats.chi2_contingency(np.array([[m_d, m_nd], [o_d, o_nd]]))
    rows.append({'feature': name,
                 'miss_diabetic': m_d, 'miss_not': m_nd,
                 'obs_diabetic' : o_d, 'obs_not' : o_nd,
                 'chi2': round(chi2, 2), 'p_value': round(p, 4)})

print('Is missingness associated with outcome?')
pd.DataFrame(rows).set_index('feature')

## 6. Model overview

In [ ]:
theta_hat, _ = train(Phi[train_idx], y[train_idx])
test_scores  = model_score(Phi[test_idx], theta_hat)
train_acc    = (predict(Phi[train_idx], theta_hat) == y[train_idx]).mean()
test_acc     = (predict(Phi[test_idx],  theta_hat) == y[test_idx]).mean()

denied_with_missing = [i for i in test_idx
                       if predict(Phi[[i]], theta_hat)[0] == -1 and Xi[i].sum() > 0]

print(f'phi shape                    : {Phi.shape}  (n x 2d)')
print(f'theta shape                  : {theta_hat.shape}  (2d+1 with bias)')
print(f'train acc                    : {train_acc:.3f}')
print(f'test acc                     : {test_acc:.3f}')
print(f'denied (test) with missing   : {len(denied_with_missing)}')
print()
print('Test score distribution:')
for label, name in [(-1, 'diabetic (denied)'), (1, 'not diabetic (approved)')]:
    s = test_scores[y[test_idx] == label]
    print(f'  {name:<25}  mean={s.mean():+.3f}  std={s.std():.3f}  '
          f'min={s.min():+.3f}  max={s.max():+.3f}')

## 7. Learned weights θ̂

In [9]:
d    = len(feature_names)
w_x  = theta_hat[:d]
w_xi = theta_hat[d:2*d]
bias = theta_hat[-1]

print(f'bias: {bias:.4f}')
df_w = pd.DataFrame({
    'w_x'    : w_x.round(4),
    'w_xi'   : w_xi.round(4),
    '|w_x|'  : np.abs(w_x).round(4),
}, index=feature_names)
df_w.sort_values('|w_x|', ascending=False)

bias: -1.0847


,w_x,w_xi,|w_x|
Glucose,1.1364,0.1314,1.1364
BMI,0.7069,-0.5927,0.7069
Age,0.3733,0.0000,0.3733
Pregnancies,0.2254,0.0000,0.2254
DiabetesPedigreeFunction,0.2245,0.0000,0.2245
BloodPressure,-0.1766,0.7147,0.1766
Insulin,-0.1212,0.5125,0.1212
SkinThickness,0.0875,-0.1319,0.0875
